In [113]:
# ========== 导入：网页抓取 + 本地 Ollama（OpenAI 兼容）摘要所需库 ==========

# 导入标准库 os：读环境变量（本练习客户端手写 api_key，os 仍保留原导入）
import os
# 导入 requests：用 HTTP GET 拉取网页 HTML
import requests
# 从 dotenv 导入 load_dotenv：课程常见写法；本文件后续未显式调用，保留原导入
from dotenv import load_dotenv
# 导入 bs4 包本身（与下一行 BeautifulSoup 并存，保留原样）
import bs4
# 从 bs4 导入 BeautifulSoup：解析 HTML 树
from bs4 import BeautifulSoup
# 导入 lxml：作为 BeautifulSoup 的解析器后端之一
import lxml
# 从 IPython.display 导入 Markdown / display：在笔记本里渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：后面会把 base_url 指到本地 Ollama
from openai import OpenAI


In [115]:
# ========== 创建 OpenAI 兼容客户端：指向本地 Ollama ==========

# base_url 指向 Ollama 的 /v1；api_key 对本地 Ollama 多为占位字符串 'ollama'
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


In [116]:
# ========== 请求头：伪装成常见浏览器，降低被站点拦截的概率 ==========

# User-Agent 等头信息：让请求更像真实浏览器，减少 block / captcha / 403
headers={
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
} 
# 一个名为 header 的字典，这样我们就可以获取与用户相同的 html 代码，并避免块、验证码和 error403


In [119]:
# ========== Website 类：下载页面 → 去噪 → 抽出标题与正文文本 ==========

class Website:
    # 构造：传入 url，立刻抓取并解析
    def __init__(self,url):
        # 保存原始 URL
        self.url=url
        # GET 网页；带上 headers；timeout=30 秒防止挂死
        response= requests.get(url,headers=headers,timeout=30)
        # 用 lxml 解析响应字节，得到 BeautifulSoup 文档树
        soup=BeautifulSoup(response.content,'lxml')
        # 取 <title> 文本；没有标题则用占位英文（保留原字符串）
        self.title=soup.title.string if soup.title else "No title found"#scraping the content
        # 删除 body 里脚本/样式/图片/输入框等无关节点，降低噪声
        for irrelevant in soup.body(["script", "style", "img", "input"]):#cleaning the content
            irrelevant.decompose()
            # 使用 Beautiful soup 的 .get_text() 方法
            # using .get_text() method of Beautiful soup
        # 抽出正文：换行分隔块，strip=True 去掉首尾空白
        self.text = soup.body.get_text(separator="\n", strip=True)#creating space between different lines and removing leading whitespaces by strip=true


In [121]:
# ========== 试抓一条娱乐站：打印标题与正文，确认爬虫可用 ==========

# 实例化：Times of India 娱乐频道（URL 保持原样）
gossip= Website("https://timesofindia.indiatimes.com/entertainment")
# 打印页面标题
print(gossip.title)
# 打印清洗后的正文文本
print(gossip.text)


Latest and Trending Entertainment News, Celebrity News, Movie News, Breaking News | Entertainment - Times of India
Sign In
TOI
Go to
TOI
Etimes
home
cinema
news
movie reviews
movie listings
box office
anime
previews
did you know
videos
showtimes
blogs
awards
News
entertainment
Trending
Javed Akhtar
Diljit Dosanjh
Jaideep Ahlawat
Karisma Kapoor
Gauri Khan
Blake Lively
Trisha Krishnan
Kuberaa Box Office Collection
Sitaare Zameen Par Box Office Collection
Housefull 5
Kuberaa Movie Review
Sitaare Zameen Par Movie Review
Javed Akhtar
Diljit Dosanjh
Jaideep Ahlawat
Karisma Kapoor
Gauri Khan
Blake Lively
Trisha Krishnan
Kuberaa Box Office Collection
Sitaare Zameen Par Box Office Collection
Housefull 5
Kuberaa Movie Review
Sitaare Zameen Par Movie Review
Javed Akhtar
Diljit Dosanjh
Jaideep Ahlawat
Karisma Kapoor
Gauri Khan
Blake Lively
Trisha Krishnan
Kuberaa Box Office Collection
Sitaare Zameen Par Box Office Collection
Housefull 5
Kuberaa Movie Review
Sitaare Zameen Par Movie Review
Sudhansh

In [123]:
# ========== system prompt：时尚 / 娱乐摘要助手人设（英文指令保留）==========

# 系统提示：语气像生活方式杂志的流行文化编辑；聚焦潮流、明星、八卦
system_prompt = """
You are a stylish and culturally aware assistant who specializes in summarizing and discussing fashion trends, celebrity style, entertainment news, and television gossip.

You stay updated on Hollywood, Bollywood, and the television world—including celebrity rumors, drama, reality TV updates, show recaps, and behind-the-scenes stories.

When summarizing content, be engaging, concise, and insightful. Focus on what's trending, who's wearing what, and what everyone is talking about in fashion and entertainment. Maintain a fun yet informative tone, like a pop culture expert writing for a lifestyle magazine.

If content includes TV gossip, highlight key rumors, casting updates, fan reactions, and noteworthy moments from popular shows.
"""


In [125]:
# ========== 拼 user prompt：把网站标题 + 正文塞进摘要任务模板 ==========

# 入参是 Website 实例；返回发给模型的 user 侧长文本（Markdown 摘要要求写在英文里，勿改译）
def user_prompt_for(website):
    # f-string：嵌入 title 与 text
    user_prompt = f"""The following text is extracted from a website titled: "{website.title}".

Please analyze this content and provide a short and engaging summary in **Markdown format**.

If the page contains:
- 🧵 Fashion trends: mention standout styles, designers, or events.
- 🗣️ TV gossip: highlight any drama, casting news, or fan reactions.
- 🎬 Celebrity updates (Hollywood/Bollywood): include relevant quotes, fashion moments, or event mentions.
- 📺 Show recaps: summarize what happened and any major twists.

Keep the summary clear, fun, and informative. Use bullet points if multiple themes appear. If there is no meaningful content, say: *“No relevant summary could be generated.”*

Website Content:
{website.text}
"""
    # 返回拼好的 user 提示词
    return user_prompt


In [127]:
# ========== 预览 user prompt：确认标题与正文已正确嵌入 ==========

# 对刚才的 gossip 页面打印将要发送的 user 内容（可能很长）
print(user_prompt_for(gossip))


The following text is extracted from a website titled: "Latest and Trending Entertainment News, Celebrity News, Movie News, Breaking News | Entertainment - Times of India".

Please analyze this content and provide a short and engaging summary in **Markdown format**.

If the page contains:
- 🧵 Fashion trends: mention standout styles, designers, or events.
- 🗣️ TV gossip: highlight any drama, casting news, or fan reactions.
- 🎬 Celebrity updates (Hollywood/Bollywood): include relevant quotes, fashion moments, or event mentions.
- 📺 Show recaps: summarize what happened and any major twists.

Keep the summary clear, fun, and informative. Use bullet points if multiple themes appear. If there is no meaningful content, say: *“No relevant summary could be generated.”*

Website Content:
Sign In
TOI
Go to
TOI
Etimes
home
cinema
news
movie reviews
movie listings
box office
anime
previews
did you know
videos
showtimes
blogs
awards
News
entertainment
Trending
Javed Akhtar
Diljit Dosanjh
Jaideep Ahl

In [129]:
# ========== 组装 messages：system 人设 + user 页面内容 ==========

# 返回 OpenAI Chat Completions 所需的两条消息
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]


In [146]:
# ========== summarize：抓取 URL → 调本地 llama3.2 → 返回摘要文本 ==========

# 入口函数：给定网址，返回模型生成的摘要字符串
def summarize(url):
    # 先爬取并清洗页面
    website = Website(url)
    # 非流式 chat.completions；model 固定为本地 llama3.2
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    # 取出助手回复正文
    return response.choices[0].message.content


In [ ]:
# ========== 直接调用 summarize：返回值会显示在单元格输出里 ==========

# 对同一娱乐频道 URL 做摘要（需本机 Ollama 已运行且已拉取 llama3.2）
summarize("https://timesofindia.indiatimes.com/entertainment")


In [139]:
# ========== display_summary：摘要结果用 Markdown 漂亮展示 ==========

# 包装函数：先 summarize，再 display(Markdown(...))
def display_summary(url):
    # 拿到纯文本摘要
    summary = summarize(url)
    # 在笔记本中渲染为 Markdown
    display(Markdown(summary))


In [ ]:
# ========== 运行展示：娱乐站时尚/八卦风摘要 ==========

# 端到端：爬取 → 提示词 → llama3.2 → Markdown 显示
display_summary("https://timesofindia.indiatimes.com/entertainment")
